In [1]:
import os
os.chdir('/Users/taruni/Desktop/virality-prediction-ml-2026')
print('Working directory:', os.getcwd())

Working directory: /Users/taruni/Desktop/virality-prediction-ml-2026


# 03 — Modeling
Train and compare: Logistic Regression, KNN, SVM, Decision Tree, Random Forest, XGBoost

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load master dataset
master_df = pd.read_csv('data/processed/master.csv')
print('Loaded:', master_df.shape)
print('Viral %:', master_df['viral'].mean().round(3))

Loaded: (90032, 35)
Viral %: 0.2


In [4]:
# Fix day_of_week — convert string days to numbers
day_map = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 
           'Friday': 4, 'Saturday': 5, 'Sunday': 6}

master_df['day_of_week'] = master_df['day_of_week'].map(day_map).fillna(master_df['day_of_week'])
master_df['day_of_week'] = pd.to_numeric(master_df['day_of_week'], errors='coerce').fillna(0).astype(int)

print('day_of_week sample:', master_df['day_of_week'].unique()[:10])

day_of_week sample: [5 4 3 1 6 0 2]


In [5]:
# ── TRAIN/TEST SPLIT ──

# Features = everything except viral and engagement_rate
# (engagement_rate would be cheating — it's how we defined viral!)
X = master_df.drop(columns=['viral', 'engagement_rate'])
y = master_df['viral']

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)

print('Training set:', X_train.shape)
print('Test set:    ', X_test.shape)
print()
print('Viral % in train:', y_train.mean().round(3))
print('Viral % in test: ', y_test.mean().round(3))

Training set: (72025, 33)
Test set:     (18007, 33)

Viral % in train: 0.2
Viral % in test:  0.2


In [6]:
# ── MODEL 1: LOGISTIC REGRESSION (baseline) ──
print("Training Logistic Regression...")

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:, 1]

print("\n--- Logistic Regression Results ---")
print(classification_report(y_test, lr_preds, target_names=['Not Viral', 'Viral']))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}")

Training Logistic Regression...

--- Logistic Regression Results ---
              precision    recall  f1-score   support

   Not Viral       0.80      1.00      0.89     14413
       Viral       0.00      0.00      0.00      3594

    accuracy                           0.80     18007
   macro avg       0.40      0.50      0.44     18007
weighted avg       0.64      0.80      0.71     18007

ROC-AUC: 0.6089


In [7]:
# ── MODEL 1 FIXED: LOGISTIC REGRESSION with class balancing ──
print("Training Logistic Regression (balanced)...")

lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:, 1]

print("\n--- Logistic Regression Results ---")
print(classification_report(y_test, lr_preds, target_names=['Not Viral', 'Viral']))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}")

Training Logistic Regression (balanced)...

--- Logistic Regression Results ---
              precision    recall  f1-score   support

   Not Viral       0.83      0.53      0.65     14413
       Viral       0.23      0.57      0.33      3594

    accuracy                           0.54     18007
   macro avg       0.53      0.55      0.49     18007
weighted avg       0.71      0.54      0.59     18007

ROC-AUC: 0.6084
